In [1]:
from pathlib import Path
import yaml
import numpy as np
from astropy import stats
from astropy.time import Time
from astropy.table import Table, MaskedColumn
from kpf_etc.etc import kpf_photon_noise_estimate

import matplotlib.pyplot as plt

data_file = 'data/Feb2026_withInstZPs.csv'
t = Table.read(data_file, format='ascii.csv')

In [2]:
EMZPfile = Path('data/ExposureMeterZeroPoints.yaml')
with open(EMZPfile, 'r') as f:
    EMZeroPoint = yaml.safe_load(f)
EMZeroPoint

{1: 21.296749750531042,
 2: 21.802875631774594,
 3: 21.738583308861333,
 4: 21.385165124935046,
 'SUM': 23.06521519492087}

In [3]:
ZPfile = Path('data/InstrumentZeroPoints.yaml')
with open(ZPfile, 'r') as f:
    ZeroPoint = yaml.safe_load(f)
ZeroPoint

{'452': 13.664278495210038,
 '548': 14.964690898633878,
 '652': 15.329155098901115,
 '747': 15.466755734095996,
 '852': 15.073233551565435}

In [4]:
EMbinfile = Path('data/ExpMeterBins.yaml')
with open(EMbinfile, 'r') as f:
    best_EMbin = yaml.safe_load(f)
best_EMbin

{'452': 1, '548': 2, '652': 3, '747': 4, '852': 4}

# Build an ETC From the "Nominal" Zero Point Value

In [5]:
def ZPETC(Gmag, exptime, instwav='652', EMbin=2):
    assert instwav in ZeroPoint.keys()
    assert EMbin in EMZeroPoint.keys()
    photons = float(exptime*10**((Gmag - ZeroPoint[instwav])/-2.5))
    EM_ADU = float(exptime*10**((Gmag - EMZeroPoint[EMbin])/-2.5))
    return photons, EM_ADU


def EMThreshold(photons, instwav='652', EMbin=2):
    assert instwav in ZeroPoint.keys()
    assert EMbin in EMZeroPoint.keys()
    ZPdiff = ZeroPoint[instwav] - EMZeroPoint[EMbin]
    EM_ADU = photons*10**(-ZPdiff/2.5)
    return EM_ADU

In [6]:
photons, EMADU = ZPETC(10, 60, instwav='747', EMbin=3)
photons, EMADU

(9222.60363288268, 2975668.6802112367)

In [7]:
EMADU2 = EMThreshold(photons, instwav='747', EMbin=3)
EMADU2/EMADU

1.000000000000001